In [ ]:
Heres the code with added docstrings and inline comments for clarity:

```python
import re
import pandas as pd
import json
from collections import defaultdict
import os

# List of chat files to process
chat_files = [
    "_chat.txt",
    "_chat1.txt",
    "_chat2.txt",
    "_chat3.txt",
    "_chat4.txt"
]

# Directory where the processed datasets will be saved
output_dir = "contact_datasets"
os.makedirs(output_dir, exist_ok=True)  # Create directory if it doesn't exist

# Regular expression pattern to extract chat details
pattern = r"\[(\d{2}/\d{2}/\d{4}), (\d{2}:\d{2}:\d{2})\] ([^:]+): (.*)"

# Name of the user extracting the chat, adjust as necessary
MY_NAME = "Mira JESUS Babygirl💕💗💖💓💝❤️‍🔥"

for file in chat_files:
    print(f"\nProcessing {file}")

    with open(file, "r", encoding="utf-8") as f:
        text = f.read()

    # Find all matches for the defined pattern in the text
    matches = re.findall(pattern, text)

    messages = []
    # Extract date, time, sender, and message from each match
    for date, time, sender, msg in matches:
        messages.append({
            "datetime": f"{date} {time}",
            "sender": sender.strip(),
            "message": msg.strip()
        })

    # Create and sort DataFrame by datetime
    df = pd.DataFrame(messages)
    df = df.sort_values("datetime")

    # Check if the message is sent by the user
    df['is_user'] = df['sender'] == MY_NAME

    # Filter contacts that are not the user
    contacts = df[df['sender'] != MY_NAME]['sender'].unique()

    if len(contacts) != 1:
        # Warn if more than one contact exists in a single file
        print(f"⚠ Warning: {file} contains multiple contacts:", contacts)
        continue

    contact = contacts[0]
    print("Detected contact:", contact)

    pairs = []

    # Pair incoming and outgoing message
    for i in range(len(df) - 1):
        incoming = df.iloc[i]
        outgoing = df.iloc[i + 1]

        if incoming['sender'] != MY_NAME and outgoing['sender'] == MY_NAME:
            pairs.append({
                "input": incoming['message'],
                "output": outgoing['message']
            })

    # Create a safe filename for the contact by replacing non-alphanumeric characters
    safe_name = re.sub(r"[^a-zA-Z0-9]+", "_", contact)
    filename = f"{output_dir}/dataset_{safe_name}.jsonl"

    # Write paired messages to a JSONL format file
    with open(filename, "w", encoding="utf-8") as f:
        for p in pairs:
            f.write(json.dumps(p) + "\n")

    print(f"Saved {len(pairs)} pairs -> {filename}")

print("\nDone! Per-contact datasets generated cleanly.")

# Load pairs from a specific contact dataset
file_path = "contact_datasets/dataset_Maryann.jsonl"

pairs = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        pairs.append(json.loads(line))

df = pd.DataFrame(pairs)
print("Loaded examples:", len(df))
df.head()

# Define constants
CONTACT_DATA_DIR = "contact_datasets"
CONFIDENCE_THRESHOLD = 0.50
AUTO_SEND_DELAY = 10

# Lists of unwanted inputs and outputs for filtering
unwanted_inputs = [
    "‎audio omitted",
    "‎image omitted",
    "‎video omitted",
    "‎Voice call",
    "‎Video call",
    "‎sticker omitted"
]

unwanted_outputs = [
    "‎audio omitted",
    "‎image omitted",
    "‎video omitted",
    "‎Voice call, No answer",
    "‎Video call, No answer",
    "‎sticker omitted"
]


def ensure_contact_memory(contact, memory):
    """Ensure a contact has a memory DataFrame structure initialized."""
    if contact not in memory:
        memory[contact] = pd.DataFrame(
            columns=["input", "output", "embeddings"]
        )
    return memory

def load_all_contact_memory():
    """Load all contact memory from JSONL files in the CONTACT_DATA_DIR."""
    memory = {}

    # Ensure the contact data directory exists
    if not os.path.exists(CONTACT_DATA_DIR):
        os.makedirs(CONTACT_DATA_DIR)

    # Iterate over files in the contact data directory
    for file in os.listdir(CONTACT_DATA_DIR):
        if not file.endswith(".jsonl"):
            continue

        contact = file.replace("dataset_","").replace(".jsonl", "")
        path = os.path.join(CONTACT_DATA_DIR, file)

        # Load the data into a DataFrame
        df = pd.read_json(path, lines=True)

        # Filter out unwanted data entries
        df = df[df["input"].notna() & df["output"].notna()]
        df = df[~df["input"].str.lower().str.contains("|".join(unwanted_inputs), na=False)]
        df = df[~df["output"].str.lower().str.contains("|".join(unwanted_outputs), na=False)]

        df.reset_index(drop=True, inplace=True)

        # Compute embeddings if not present or incomplete
        if "embeddings" not in df.columns or df["embeddings"].isna().any():
            df["embeddings"] = df["input"].apply(get_embedding)

        memory[contact] = df

    return memory

import openai
import numpy as np

openai.api_key = ""

def get_embedding(text, model="text-embedding-3-small"):
    """Get the embedding of a given text using OpenAI embedding model."""
    response = openai.embeddings.create(
        model=model,
        input=text
    )
    return np.array(response.data[0].embedding)


df["embeddings"] = df["input"].apply(get_embedding)

print("Embeddings created.")

memory = load_all_contact_memory()

def retrieve_similar_messages(contact, message, memory, top_k=3):
    """Retrieve top_k most similar messages from contact memory."""
    memory = ensure_contact_memory(contact, memory)
    df = memory[contact]

    if len(df) == 0:
        return pd.DataFrame(columns=["input", "output", "similarity"])

    new_emb = get_embedding(message)  # Get embedding of the new message
    embeddings = np.vstack(df["embeddings"].values)

    # Calculate similarity to existing embeddings
    sims = cosine_similarity([new_emb], embeddings)[0]
    df = df.assign(similarity=sims)

    return df.sort_values("similarity", ascending=False).head(top_k)

def compute_confidence(contact, message, memory):
    """Compute the confidence level of the best match found."""
    matches = retrieve_similar_messages(contact, message, memory)
    if len(matches) == 0:
        return 0.0
    return matches["similarity"].mean()

def generate_reply(contact, message, memory):
    """Generate a reply based on previous style examples."""
    examples = retrieve_similar_messages(contact, message, memory)

    examples_text = ""
    for _, row in examples.iterrows():
        examples_text += f"user: {row['input']}\nAssistant: {row['output']}\n\n"

    prompt = f"""
reply on behalf of the user.

Previous style examples with {contact}:
{examples_text}

Incoming message:
{message}

Reply naturally:
"""

    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    
    return response.choices[0].message.content.strip()

# Words/phrases used to detect a user's specific style
style_markers = [
    "chai",
    "no dey",
    "wetin",
    "baby",
    "lol",
    "😂",
    "😊",
    "❤️",
    "nah",
    "abeg",
    "omo",
    "guyyyy",
    "as how",
    "purrrr",
    "duhhhh",
    "money",
    "love"
]

# Keywords to detect if a conversation is based around a group
GROUP_KEYWORDS = ["group", "family", "team", "project", "community"]

from sklearn.metrics.pairwise import cosine_similarity
import sklearn

def detect_user_style(text, markers):
    """Detect if the text contains any style markers."""
    text_lower = text.lower()
    return any(marker in text_lower for marker in markers)

def auto_reply_engine(contact, message, memory):
    """Generate an automatic reply based on the incoming message."""
    reply = generate_reply(contact, message, memory)
    confidence = compute_confidence(contact, message, memory)
    style = detect_user_style(reply, style_markers)

    if confidence >= CONFIDENCE_THRESHOLD:
        decision = "AUTO_AFTER_10_SECONDS"
    else:
        decision = "WAIT_FOR_APPROVAL"

    return {
        "reply": reply,
        "confidence": confidence,
        "decision": decision,
        "style_detected": style
    }

def store_new_pair(contact, incoming, reply, memory):
    """Store a new pair of incoming message and its reply in contact memory."""
    memory = ensure_contact_memory(contact, memory)
    df = memory[contact]

    # Only add if the pair (incoming, reply) does not already exist
    if ((df["input"] == incoming) &  (df["output"] == reply)).any():
        return memory

    new_row = {
        "input": incoming,
        "output": reply,
        "embeddings": get_embedding(incoming)
    }

    memory[contact] = pd.concat(
        [df, pd.DataFrame([new_row])],
        ignore_index=True
    )

    return memory

def persist_contact_memory(contact, memory):
    """Persist a contact's memory to disk in JSONL format."""
    path = os.path.join(CONTACT_DATA_DIR, f"dataset_{contact}.jsonl")
    memory[contact][["input", "output"]].to_json(
        path, orient="records", lines=True, force_ascii=False
    )

def maybe_learn(contact, incoming, reply, memory, decision):
    """Decide if a new (incoming, reply) pair should be learned and stored."""
    combined = f"{incoming} {reply}".lower()
    if any(x in combined for x in unwanted_inputs + unwanted_outputs):
        return memory

    if decision in ["AUTO_AFTER_10_SECONDS", "MANUAL_APPROVAL"]:
        memory = store_new_pair(contact, incoming, reply, memory)
        persist_contact_memory(contact, memory)

    return memory

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import pyperclip

def get_unread_chats():
    """Retrieve unread chat elements from the WhatsApp web interface."""
    chats = driver.find_elements(By.XPATH, '//div[@role="row"]')
    unread = []

    for chat in chats:
        try:
            # Locate the green unread badge (number)
            chat.find_element(
                By.XPATH,
                './/span[contains(@class,"unread")] | .//span[contains(@aria-label,"unread")]'
            )
            unread.append(chat)
        except:
            pass

    return unread

def open_chat(contact):
    """Open a chat with the given contact."""
    search = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located(
            (By.XPATH, '//div[@contenteditable="true"][@data-tab="3"]')
        )
    )
    search.click()
    search.clear()
    search.send_keys(contact)  # Type contact name
    time.sleep(1)
    search.send_keys(Keys.ENTER)  # Open the contact's chat

def open_chat_from_element(chat):
    """Open a chat by clicking on the chat element."""
    # Scroll chat into view and click to open
    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        chat
    )
    time.sleep(0.5)
    WebDriverWait(driver, 5).until(EC.element_to_be_clickable(chat))
    chat.click()
    time.sleep(1.5)

def read_last_incoming_message():
    """Read the last incoming message text from the chat."""
    messages = driver.find_elements(
        By.XPATH,
        '//div[contains(@class,"message-in")]//span[@dir="ltr"]'
    )

    if not messages:
        return None

    return messages[-1].text

def get_open_chat_name():
    """Retrieve the name of the currently open chat."""
    return WebDriverWait(driver, 10).until(
        EC.presence_of_element_located(
            (By.XPATH, '//header//span[@dir="auto"]')
        )
    ).text

from selenium.webdriver.common.action_chains import ActionChains

def send_message(message):
    """Send a message to the open WhatsApp chat."""
    actions = ActionChains(driver)
    actions.send_keys(message)
    actions.send_keys(Keys.ENTER)
    actions.perform()

from selenium.webdriver.common.keys import Keys

def send_message_to_whatsapp(contact, message):
    """Send a message to a specific contact on WhatsApp."""
    open_chat(contact)
    time.sleep(1.5)

    actions = ActionChains(driver)
    actions.send_keys(message)
    actions.send_keys(Keys.ENTER)
    actions.perform()

    print("✅ Message sent")

last_seen_messages = {}

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def draft_message(contact, message):
    """Draft a message in the chat with the specified contact without sending."""
    open_chat(contact)
    time.sleep(1)

    box = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable(
            (By.XPATH, '//footer//div[@contenteditable="true"]')
        )
    )
    box.click()
    box.send_keys(message)
    print("✍️ Draft typed into WhatsApp")

def draft_into_whatsapp(text):
    """Draft a message into the WhatsApp chat interface."""
    box = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable(
            (By.XPATH, '//footer//div[@contenteditable="true"]')
        )
    )
    box.click()
    box.clear()
    box.send_keys(text)
    print("✍️ Draft typed into WhatsApp input")

def capture_last_outgoing_message():
    """Capture the last outgoing message sent from the open chat."""
    messages = driver.find_elements(
        By.XPATH,
        '//div[contains(@class,"message-out")]//span[@dir="ltr"]'
    )

    if not messages:
        return None

    return messages[-1].text.strip()

from selenium.common.exceptions import (
    InvalidSessionIdException,
    WebDriverException,
    NoSuchElementException,
    StaleElementReferenceException,
)

def prime_last_seen_messages():
    """Prime a baseline with the last seen messages for active chats."""
    chats = driver.find_elements(By.XPATH, '//div[@role="row"]')[:15]

    for chat in chats:
        try:
            contact = chat.find_element(By.XPATH, './/span[@dir="auto"]').text.strip()

            preview = chat.find_element(
                By.XPATH, './/span[contains(@class,"_11JPr")]'
            ).text.strip()

            last_seen_messages[contact] = preview

        except:
            continue

    print("✅ Preview baseline primed")

# Initialize the Selenium WebDriver
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)
driver.get("https://web.whatsapp.com")
input("Scan QR, wait for chats, then press Enter")
time.sleep(8)
prime_last_seen_messages()

last_incoming_text = None

def check_for_new_message():
    """Check the open chat for any new incoming messages."""
    global last_incoming_text
    events = []

    try:
        contact = driver.find_element(
            By.XPATH, '//header//span[@dir="auto"]'
        ).text.strip()

        messages = driver.find_elements(
            By.XPATH, '//div[contains(@class,"message-in")]//span[@dir="ltr"]'
        )

        if not messages:
            return events

        latest = messages[-1].text.strip()

        if latest != last_incoming_text:
            last_incoming_text = latest
            events.append((contact, latest))

    except Exception as e:
        print("DEBUG check error:", e)

    return events

AUTO_SEND_DELAY = 10

# Keywords related to financial context
MONEY_KEYWORDS = [
    "money", "pay", "payment", "transfer",
    "send", "owe", "price", "cost",
    "account", "bank", "naira", "₦"
]

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)
driver.get("https://web.whatsapp.com")

input("✅ Scan QR code, WAIT for chats, then press Enter")

print("⏳ Stabilizing WhatsApp...")
time.sleep(10)
print("✅ WhatsApp ready")

print("🤖 WhatsApp AI auto-responder running...")

while True:
    try:
        unread_chats = get_unread_chats()

        if not unread_chats:
            time.sleep(3)
            continue

        print(f"\n🔍 Found {len(unread_chats)} unread chats")

        for chat in unread_chats:
            open_chat_from_element(chat)
            time.sleep(1)

            contact = get_open_chat_name()
            message = read_last_incoming_message()

            if not message:
                continue

            print(f"\n📩 New message from {contact}")
            print(f"➡️ {message}")

            # Skip group chats
            if "," in contact or "group" in contact.lower():
                print("👥 Group chat detected — skipping")
                continue

            # Skip shipment or system/business messages
            if "shipment" in message.lower():
                print("📦 System / business message — skipping")
                continue

            # Detect money-related messages and send a safe reply
            if any(word in message.lower() for word in MONEY_KEYWORDS):
                safe_reply = "Let me get back to you on this shortly 🙏"

                print("💰 Money-related message detected")
                print("✍️ Drafted reply:")
                print(f"➡️ {safe_reply}")

                send_message(safe_reply)
                continue

            memory = ensure_contact_memory(contact, memory)
            result = auto_reply_engine(contact, message, memory)

            confidence_pct = round(result["confidence"] * 100, 1)

            print("\n💡 AI Suggested Reply:")
            print(f"➡️ {result['reply']}")
            print(f"📊 Confidence: {confidence_pct}%")

            # Auto-sending decisions based on confidence
            if result["decision"] == "AUTO_AFTER_10_SECONDS":
                print(f"⏳ Auto-sending in {AUTO_SEND_DELAY} seconds...")
                time.sleep(AUTO_SEND_DELAY)

                send_message(result["reply"])

                memory = maybe_learn(
                    contact,
                    incoming=message,
                    reply=result["reply"],
                    memory=memory,
                    decision="AUTO_AFTER_10_SECONDS"
                )

                print("✅ Message sent & learned")

            else:
                print("✍️ Drafting reply for manual approval")
            
                draft_into_whatsapp(result["reply"])
            
                print("🛑 BOT PAUSED — waiting for you (15s)")
            
                sent = None
                start = time.time()
                LOCK_TIME = 30
            
                while time.time() - start < LOCK_TIME:
                    time.sleep(10)
            
                    sent = capture_last_outgoing_message()
                    if sent:
                        break
            
                if sent:
                    print("✅ You sent:", sent)
            
                    memory = maybe_learn(
                        contact,
                        incoming=message,
                        reply=sent,
                        memory=memory,
                        decision="MANUAL_APPROVAL"
                    )
                else:
                    print("❌ No manual send detected — skipped learning")
                break

        time.sleep(2)

    except Exception as e:
        print("⚠️ Loop error:", e)

# Memory structure for contact learning
memory[contact] = {
    "history": [...],
    "style": ...,
    "confidence": ...
}
```

In this version, docstrings and inline comments provide explanations of the logic, purpose, and flow at various points in the code.